In [ ]:
%pip install torch transformers accelerate

In [113]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# we take 3 inptuts of different lengths , tokenize, then pad them together to see if it would help

In [116]:
# First get the 3 values
a = "Hello my name is qalid and i like to "
b = "small text"
c = "The FitnessGram Pacer Test is a multistage " #aerobic capacity test that progressively gets more difficult as it continues. The 20 meter pacer test will begin in 30 seconds


In [117]:
# now from text we need to essentially convert tokenes
aToken = tokenizer.tokenize(a)
bToken = tokenizer.tokenize(b)
cToken = tokenizer.tokenize(c)
# print the tokens for each
print(aToken)
print(bToken)
print(cToken)

['Hello', 'Ġmy', 'Ġname', 'Ġis', 'Ġq', 'al', 'id', 'Ġand', 'Ġi', 'Ġlike', 'Ġto', 'Ġ']
['small', 'Ġtext']
['The', 'ĠFitness', 'Gram', 'ĠP', 'acer', 'ĠTest', 'Ġis', 'Ġa', 'Ġmult', 'ist', 'age', 'Ġ']


In [118]:
# now from tokens we need to convert to ids
token_idA = tokenizer.encode(a)
token_idB = tokenizer.encode(b)
token_idC = tokenizer.encode(c)
# now print all th tokens ids -> this essentially is a intemidiate m
print(token_idA)
print(token_idB)
print(token_idC)

[9707, 847, 829, 374, 2804, 278, 307, 323, 600, 1075, 311, 220]
[9004, 1467]
[785, 35708, 64225, 393, 9584, 3393, 374, 264, 2745, 380, 424, 220]


In [119]:
largestLen = max(len(token_idA), len(token_idB), len(token_idC))

pad_id = tokenizer.pad_token_id
if pad_id is None:
    pad_id = tokenizer.eos_token_id

newA = [pad_id] * (largestLen - len(token_idA)) + token_idA
newB = [pad_id] * (largestLen - len(token_idB)) + token_idB
newC = [pad_id] * (largestLen - len(token_idC)) + token_idC

print(newA)
print(newB)
print(newC)

[9707, 847, 829, 374, 2804, 278, 307, 323, 600, 1075, 311, 220]
[151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 9004, 1467]
[785, 35708, 64225, 393, 9584, 3393, 374, 264, 2745, 380, 424, 220]


In [120]:
# convert to tensors and bring it together
import torch
tensorA = torch.tensor(newA)
tensorB = torch.tensor(newB)
tensorC = torch.tensor(newC)

print(tensorA)
print(tensorB)
print(tensorC)


tensor([9707,  847,  829,  374, 2804,  278,  307,  323,  600, 1075,  311,  220])
tensor([151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
        151643,   9004,   1467])
tensor([  785, 35708, 64225,   393,  9584,  3393,   374,   264,  2745,   380,
          424,   220])


In [121]:
# combine the tensores into one
batch = torch.stack([tensorA,tensorB,tensorC])
print(batch)

tensor([[  9707,    847,    829,    374,   2804,    278,    307,    323,    600,
           1075,    311,    220],
        [151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643,   9004,   1467],
        [   785,  35708,  64225,    393,   9584,   3393,    374,    264,   2745,
            380,    424,    220]])


In [122]:
print(batch.shape)

torch.Size([3, 12])


In [123]:
# first we need to use attention masking for the padding 
attention_mask = (batch != pad_id).long()
# after padding we ask the model perdict a single token for each request 
with torch.no_grad():
    outputs = model(
        input_ids=batch,
        attention_mask=attention_mask
    )
# See the padding logits and the representation for it 
logits = outputs.logits
print(logits)
print(logits.shape)

tensor([[[16.1250,  8.3750,  3.6719,  ..., -2.1094, -2.1094, -2.1094],
         [ 6.2500,  5.5312,  4.3750,  ..., -0.4785, -0.4785, -0.4785],
         [ 9.7500, 11.0625,  7.0625,  ..., -0.9492, -0.9492, -0.9492],
         ...,
         [ 7.2812,  7.3438,  5.5312,  ...,  0.0250,  0.0250,  0.0250],
         [ 6.5000,  8.0625,  5.8750,  ..., -1.4297, -1.4297, -1.4297],
         [ 0.4375, -5.3750, -0.7891,  ..., -1.6328, -1.6328, -1.6328]],

        [[ 6.2500,  6.9062,  6.5000,  ..., -3.0938, -3.0938, -3.0938],
         [ 6.2500,  6.9062,  6.5000,  ..., -3.0938, -3.0938, -3.0938],
         [ 6.2500,  6.9062,  6.5000,  ..., -3.0938, -3.0938, -3.0938],
         ...,
         [ 6.2500,  6.9062,  6.5000,  ..., -3.0938, -3.0938, -3.0938],
         [ 5.9375, 10.7500,  4.8750,  ..., -2.8438, -2.8438, -2.8438],
         [ 8.6875, 12.0625,  5.0000,  ..., -3.5312, -3.5312, -3.5312]],

        [[ 3.3594,  4.8750,  4.3750,  ...,  0.2236,  0.2236,  0.2236],
         [ 6.5625,  4.0938,  3.1406,  ..., -2

In [124]:
# shows all the next likely wor
# last real position for request A
for i in range(3):
    last_pos_a = attention_mask[i].sum() - 1

    # logits for the next token
    next_logits_a = logits[i, last_pos_a, :]

    # get top 10 token IDs
    top_values, top_ids = torch.topk(next_logits_a, k=10)

    for score, token_id in zip(top_values, top_ids):
        token_text = tokenizer.decode([token_id.item()])
        print(token_id.item(), score.item(), repr(token_text))
    print("")

1486 17.375 ' play'
18 17.125 '3'
1281 17.125 ' make'
3960 17.0 ' learn'
1414 16.5 ' know'
653 16.5 ' do'
2548 16.25 ' ask'
1936 16.25 ' build'
387 15.9375 ' be'
2038 15.875 ' code'

82 10.1875 's'
198 9.8125 '\n'
220 9.5625 ' '
16 9.0 '1'
9370 8.875 '的'
271 8.75 '\n\n'
77 8.6875 'n'
17 8.6875 '2'
18493 8.625 '在'
20412 8.375 '是'

17 27.75 '2'
16 21.0 '1'
21 19.125 '6'
18 18.0 '3'
20 17.625 '5'
19 17.25 '4'
65892 16.5 ' _____'
30743 16.5 ' ____'
23 16.375 '8'
220 15.625 ' '



In [125]:
new_tokens = []

for i in range(3):
    # last position is the final column because we left-padded
    next_logits = logits[i, -1, :]

    # highest scoring next token
    top_values, top_ids = torch.topk(next_logits, k=1)

    new_tokens.append(top_ids[0])

# [3]
new_tokens = torch.stack(new_tokens)

# [3] -> [3, 1]
new_tokens = new_tokens.unsqueeze(1)
 
# append one generated token to every request
batch = torch.cat([batch, new_tokens], dim=1)

print(batch.shape)

torch.Size([3, 13])


In [126]:
eos_id = tokenizer.eos_token_id

# one boolean per request
finished = torch.zeros(batch.shape[0], dtype=torch.bool)

num_steps = 50

for step in range(num_steps):

    # update mask
    attention_mask = (batch != pad_id).long()

    # run model
    with torch.no_grad():
        outputs = model(
            input_ids=batch,
            attention_mask=attention_mask
        )

    logits = outputs.logits

    # logits for last token position of each request
    next_logits = logits[:, -1, :]

    # choose highest scoring token
    next_tokens = torch.argmax(next_logits, dim=-1)

    # for requests already finished, force EOS again
    next_tokens[finished] = eos_id

    # mark newly finished requests
    finished = finished | (next_tokens == eos_id)

    # append one token per request
    batch = torch.cat(
        [batch, next_tokens.unsqueeze(1)],
        dim=1
    )

    print("step:", step)
    print("next tokens:", next_tokens)
    print("finished:", finished)

    # stop whole loop once everyone is done
    if finished.all():
        break

step: 0
next tokens: tensor([3868, 1467,   15])
finished: tensor([False, False, False])
step: 1
next tokens: tensor([  323,  3460, 72501])
finished: tensor([False, False, False])
step: 2
next tokens: tensor([  600,  1467, 44541])
finished: tensor([False, False, False])
step: 3
next tokens: tensor([1075,  271, 1598])
finished: tensor([False, False, False])
step: 4
next tokens: tensor([ 311,  785, 1273])
finished: tensor([False, False, False])
step: 5
next tokens: tensor([1486, 8513,  429])
finished: tensor([False, False, False])
step: 6
next tokens: tensor([3868, 6647, 8552])
finished: tensor([False, False, False])
step: 7
next tokens: tensor([389, 315, 288])
finished: tensor([False, False, False])
step: 8
next tokens: tensor([  847,   220, 31744])
finished: tensor([False, False, False])
step: 9
next tokens: tensor([4540,   16,  416])
finished: tensor([False, False, False])
step: 10
next tokens: tensor([ 323,   23, 5565])
finished: tensor([False, False, False])
step: 11
next tokens: ten

In [127]:
for i in range(batch.shape[0]):
    print(tokenizer.decode(batch[i], skip_special_tokens=True))
    print()

Hello my name is qalid and i like to  play games and i like to play games on my phone and i like to play games on my computer and i like to play games on my xbox and i like to play games on my ps4 and i like to play games on my pc and i like

small text medium text large text

The Great Fire of 1871

The Great Fire of 1871

by Ann McGovern

The Great Fire of 1871 is a book that tells the story of the Great Chicago Fire

The FitnessGram Pacer Test is a multistage 20-meter shuttle run test that assesses cardiorespiratory endurance. The test is based on the 20-meter shuttle run, which is a multistage fitness test that progressively gets more difficult as the test progresses. The test is performed on

